In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from executorch.exir import EdgeCompileConfig, to_edge

from torch.export import export
import matplotlib.pyplot as plt
import numpy as np

from PIL import Image


In [4]:
img = Image.open("/home/busoye_tm/Pictures/Untitled.png")
img.show()

resized_img = img.resize((28, 28)).convert('L')

# resized_img.show()
img_array = (255 - np.asarray(resized_img)) / 255


# 2. Display the array as an image
plt.imshow((img_array.reshape(28,28)), cmap='gray') # 'gray' colormap is needed for 2D arrays
plt.axis('off') # Optional: turns off axis labels
plt.show() 

FileNotFoundError: [Errno 2] No such file or directory: '/home/busoye_tm/Pictures/Untitled.png'

In [5]:
# Constants
INPUT_SIZE = 784  # 28*28 for MNIST
HIDDEN1_SIZE = 32
HIDDEN2_SIZE = 16
OUTPUT_SIZE = 10
IMAGE_SIZE = 28

In [6]:
# 1. Configuration and Hyperparameters
BATCH_SIZE = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Loading and Preprocessing
# Define transformations: Convert images to Tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard normalization for MNIST
])

# Load Datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Define DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [7]:
first_image = train_dataset[0][0]
first_image.min()

tensor(-0.4242)

In [8]:
class TinyMLPMNIST(nn.Module):
    """A small MLP for MNIST digit classification."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_SIZE, HIDDEN1_SIZE)
        self.fc2 = nn.Linear(HIDDEN1_SIZE, HIDDEN2_SIZE)
        self.fc3 = nn.Linear(HIDDEN2_SIZE, OUTPUT_SIZE)

    def forward(self, x):
        """Forward pass through the network."""
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
# 4. Define Loss Function and Optimizer
# NLLLoss is often used with log_softmax
model = TinyMLPMNIST()
crit = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr= 0.01, weight_decay= 1e-4)

In [10]:
EPOCHS = 3
LR = 0.01

In [11]:
def train(model, device, train_loader, optimizer, epoch):
    """Train the model for one epoch. Returns (model, list_of_losses)."""
    model.train()
    model = model.to(device)
    losses = []

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = crit(output, target)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if batch_idx % 100 == 0:
            print(
                f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                f"({100. * batch_idx / len(train_loader):.0f}%)]\t"
                f"Loss: {loss.item():.6f}"
            )

    return model, losses


def test(model, device, test_loader):
    """Evaluate the model on the test set and print accuracy."""
    model.eval()
    model = model.to(device)
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += crit(output, target).item()           # sum batch losses
            pred = output.argmax(dim=1, keepdim=True)          # get index of max logit
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)

    print(
        f"\nTest set: Average loss: {test_loss:.4f}, "
        f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n"
    )


In [12]:
for epoch in range(1, EPOCHS + 1):
    model, losses = train(model, DEVICE, train_loader, optimizer, epoch)
    test(model, DEVICE, test_loader)

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.284712
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.406875
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.512997
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.139871
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.167361
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.291493
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.074875
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.430099
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.115449
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.185780

Test set: Average loss: 0.2068, Accuracy: 9378/10000 (93.78%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.179835
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.117377
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.153265
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.183366
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.298222
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.091049
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.051202
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.302091
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.05218

In [13]:
img = Image.open("/home/busoye_tm/Pictures/Untitled.png")
resized_img = img.resize((28, 28)).convert('L')
# rotated_img = img.rotate(45)

# resized_img.show()
img_array = (255 - np.asarray(resized_img)) / 255


# 2. Display the array as an image
plt.imshow((img_array.reshape(28,28)), cmap='gray') # 'gray' colormap is needed for 2D arrays
plt.axis('off') # Optional: turns off axis labels
plt.show() 

img_array = transform(img_array).to(dtype= torch.float32)
img_array.shape

FileNotFoundError: [Errno 2] No such file or directory: '/home/busoye_tm/Pictures/Untitled.png'

In [14]:
model(first_image).argmax()

tensor(5)

In [15]:
str_c = "float array[28][28] = {"
for x in (first_image[0]):
    str_c += "\n{"
    for y in x:
        str_c += str(y.item()) + ','
    str_c += "},"

str_c += "\n};"
print(str_c)

float array[28][28] = {
{-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,},
{-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.4242129623889923,-0.42421296

In [16]:
param_count = sum(p.numel() for p in model.parameters())
print(f"\n📊 Model parameters: {param_count:,}")


print("📦 Exporting...")
with torch.no_grad():
    exported_program = export(model, (train_dataset[0][0],))

print("⚙️ Converting to ExecuTorch...")
edge_config = EdgeCompileConfig(_check_ir_validity=False)
edge_manager = to_edge(exported_program, compile_config=edge_config)
et_program = edge_manager.to_executorch()

# Save with error handling
filename = "balanced_tiny_mlp_mnist.pte"
print(f"💾 Saving {filename}...")
try:
    with open(filename, "wb") as f:
        f.write(et_program.buffer)

    model_size_kb = len(et_program.buffer) 
    print("✅ Export complete!")
    print(f"📁 Model size: {model_size_kb:.1f} B")
except IOError as e:
    print(f"❌ Failed to save model: {e}")


📊 Model parameters: 25,818
📦 Exporting...
⚙️ Converting to ExecuTorch...
💾 Saving balanced_tiny_mlp_mnist.pte...
✅ Export complete!
📁 Model size: 106216.0 B


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/tinyml/tinyml_executorch/lib/python3.12/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


In [ ]:
train_dataset[0][0].shape

torch.Size([1, 28, 28])

In [ ]:
img_array.shape

torch.Size([1, 28, 28])

In [ ]:
import torch
from executorch.runtime import Runtime

runtime = Runtime.get()

# input_tensor = torch.randn(1, 3, 32, 32)
program = runtime.load_program("balanced_tiny_mlp_mnist.pte")
method = program.load_method("forward")
outputs = method.execute([img_array])
print(outputs[0],outputs[0].argmax())

[program.cpp:153] InternalConsistency verification requested but not available


tensor([[ 5.8354, -8.3403, -2.0681, -1.9625, -2.6718,  1.6592,  7.2584, -3.4788,
          0.0957, -1.7409]]) tensor(6)


In [ ]:
import serial
def send_microcontroller(num_buf):
    s = serial.Serial(port='/dev/ttyACM0', baudrate=115200, timeout=1)
    s.reset_input_buffer()
    s.write(b'\x10')
    print(s.read())
    buf = num_buf.reshape(-1).numpy()
    buf_len = len(buf)
    sent_len =0
    while buf_len > sent_len:
        end_len = sent_len + 16
        if end_len > buf_len:
            end_len = buf_len
        cur_buf = bytearray(buf[sent_len: end_len].tobytes())
        s.write(cur_buf)
        if (s.read() == b'\x7f'):
            sent_len += 16
        else:
            raise ValueError("")
    s.close()

In [ ]:
send_microcontroller(img_array)

SerialException: [Errno 2] could not open port /dev/ttyACM0: [Errno 2] No such file or directory: '/dev/ttyACM0'